<a href="https://colab.research.google.com/github/guillermoLop/Data-Engineering/blob/main/GuillermoLopez_TP1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP1 - Pipeline de extracción desde API hacia Delta Lake

Este notebook desarrolla un primer pipeline de Data Engineering en Python.

El objetivo es extraer datos desde una API, convertirlos en DataFrames de pandas y almacenarlos en formato Delta Lake dentro de una estructura tipo data lake.

Para este trabajo se utiliza la API de TMDb, una API orientada a información de películas. Se implementan dos tipos de extracción:

- **Extracción full**: se obtiene el catálogo de géneros de películas, considerado un dataset relativamente estático.
- **Extracción incremental**: se obtiene el listado de películas en tendencia diaria, considerado un dataset temporal que puede cambiar con el paso del tiempo.

Los datos se almacenan en una capa `bronze`, ya que en esta primera etapa se guardan crudos o con transformaciones mínimas.


## Decisiones del proyecto

Para este TP se eligió la API de TMDb porque permite trabajar con endpoints simples, documentados y relacionados con películas.

Se seleccionaron dos endpoints de la misma API:

1. `/genre/movie/list`: devuelve el listado de géneros de películas. Se utiliza como extracción full porque es un catálogo relativamente estable.
2. `/trending/movie/day`: devuelve películas en tendencia diaria. Se utiliza como extracción incremental porque su contenido puede variar diariamente.

El almacenamiento se realiza en formato Delta Lake, organizando los datos en una estructura de carpetas similar a un data lake.

In [ ]:
!pip install pandas requests deltalake pyarrow

## Importación de librerías

En esta sección se importan las librerías que se utilizarán durante el pipeline.

- `os`: se utiliza para crear directorios automáticamente.
- `datetime`: se utiliza para registrar fecha y hora de extracción.
- `requests`: se utiliza para consultar la API.
- `pandas`: se utiliza para trabajar con DataFrames.
- `write_deltalake`: se utiliza para escribir datos en formato Delta Lake.
- `DeltaTable`: se utiliza para leer y validar las tablas Delta generadas.

In [ ]:
import os
from datetime import datetime

import requests
import pandas as pd
from deltalake.writer import write_deltalake
from deltalake import DeltaTable

## Configuración de la API

En esta sección se configuran los datos necesarios para conectarse a la API de TMDb.

La variable `BASE_URL` contiene la URL base de la API.  
La variable `HEADERS` contiene la información de autorización necesaria para realizar las consultas.

Por seguridad, el token real no debería quedar expuesto públicamente en el notebook o en GitHub. Para ejecutar el notebook, se debe reemplazar `"PEGAR_TOKEN_AQUI"` por un token válido de TMDb, o bien utilizar un mecanismo seguro como los Secrets de Google Colab.

In [ ]:
TMDB_TOKEN = "PEGAR_TOKEN_AQUI"

BASE_URL = "https://api.themoviedb.org/3"

HEADERS = {
    "Authorization": f"Bearer {TMDB_TOKEN}",
    "accept": "application/json"
}

## Creación de estructura tipo Data Lake

En esta celda se define la estructura de carpetas donde se almacenarán los datos en formato Delta Lake.

Se utiliza una capa `bronze`, ya que los datos se guardan crudos o con transformaciones mínimas.

La estructura generada será:

```plaintext
data_lake/
└── bronze/
    └── tmdb/
        ├── movie_genres/
        └── trending_movies_daily/

In [ ]:
BASE_PATH = "data_lake/bronze/tmdb"

PATH_GENRES = f"{BASE_PATH}/movie_genres"
PATH_TRENDING = f"{BASE_PATH}/trending_movies_daily"

os.makedirs(PATH_GENRES, exist_ok=True)
os.makedirs(PATH_TRENDING, exist_ok=True)

print("Directorios creados correctamente.")

In [ ]:
def get_json(endpoint, params=None):
    """
    Consulta un endpoint de TMDb y devuelve la respuesta en formato JSON.
    """
    url = f"{BASE_URL}{endpoint}"

    response = requests.get(
        url,
        headers=HEADERS,
        params=params,
        timeout=10
    )

    if response.status_code != 200:
        raise Exception(
            f"Error al consultar API. Status: {response.status_code}. Respuesta: {response.text}"
        )

    return response.json()

In [ ]:
genres_json = get_json(
    "/genre/movie/list",
    params={"language": "es-AR"}
)

df_genres = pd.DataFrame(genres_json["genres"])

df_genres["fecha_extraccion"] = datetime.now().date().isoformat()
df_genres["fuente"] = "tmdb_api"
df_genres["endpoint"] = "/genre/movie/list"

df_genres.head()

In [ ]:
write_deltalake(
    PATH_GENRES,
    df_genres,
    mode="overwrite"
)

print("Extracción full guardada en Delta Lake.")

In [ ]:
now = datetime.now()

trending_json = get_json(
    "/trending/movie/day",
    params={"language": "es-AR"}
)

df_trending = pd.DataFrame(trending_json["results"])

df_trending["fecha_extraccion"] = now.date().isoformat()
df_trending["hora_extraccion"] = now.strftime("%H")
df_trending["fuente"] = "tmdb_api"
df_trending["endpoint"] = "/trending/movie/day"

df_trending.head()

In [ ]:
write_deltalake(
    PATH_TRENDING,
    df_trending,
    mode="append",
    partition_by=["fecha_extraccion", "hora_extraccion"]
)

print("Extracción incremental guardada en Delta Lake.")

In [ ]:
!find data_lake -maxdepth 5 -type d

In [ ]:
from deltalake import DeltaTable

dt_genres = DeltaTable(PATH_GENRES)
df_genres_check = dt_genres.to_pandas()

df_genres_check.head()

In [ ]:
dt_trending = DeltaTable(PATH_TRENDING)
df_trending_check = dt_trending.to_pandas()

df_trending_check.head()